___
<h3> Read the data structure from the JSON file, that is a result of raw file preprocessing with the <a href='data_preprocessing.ipynb'>pre-processing notebook </a>.</h3>

___

In [ ]:
import os
import json

data_processed_folder = "data_processed"
json_file = "pages_with_tables.json"
json_path = os.path.join(data_processed_folder, json_file)

with open(json_path, "r", encoding="utf-8") as f:
    documents = json.load(f)

print(f"Loaded {len(documents)} documents")

all_pages = [
    {
        "document_name": doc["document_name"],
        "page_num": page.get("page_num"),
        "text": page.get("text", ""),
        "tables": page.get("tables", [])
    }
    for doc in documents
    for page in doc.get("pages", [])
]

print(f"Loaded {len(all_pages)} total pages")

pages_with_tables = [p for p in all_pages if p["tables"]]
print(f"Pages with tables: {len(pages_with_tables)}")

all_tables = [
    {
        "document_name": page["document_name"],
        "page_num": page["page_num"],
        "table_index": idx + 1,
        "table": table
    }
    for page in all_pages
    for idx, table in enumerate(page.get("tables", []))
]

print(f"Total parsed tables: {len(all_tables)}")

____
<h3> We build signatures to figure out how many unique tables we have, to determine the strategy for grouping the data </h3>

____

In [ ]:
import hashlib
from collections import defaultdict
import os
import json
import re

def signature_id(sig_tuple):
    return hashlib.sha1("||".join(sig_tuple).encode("utf-8")).hexdigest()

def header_signature(headers: list) -> tuple:
    """Returns a normalized tuple suitable for use as a dict key / hash input."""
    result = []
    for idx, h in enumerate(headers):
        h = re.sub(r"\s+", "_", h.strip().lower())
        h = re.sub(r"[^a-z0-9_]+", "", h)
        result.append(h if h else f"col_{idx+1}")
    return tuple(result)
# Build canonical structure: one row per unique header signature
# with metadata + one representative table sample
unique_headers_catalog = {}

for doc in documents:
    doc_name = doc.get("document_name", "")
    for page in doc.get("pages", []):
        page_num = page.get("page_num")
        for t_idx, t in enumerate(page.get("tables", []), start=1):
            raw_headers = t.get("headers", [])
            sig = header_signature(raw_headers)  # your function
            if not sig:
                continue

            sid = signature_id(sig)
            rows = t.get("rows", [])

            if sid not in unique_headers_catalog:
                unique_headers_catalog[sid] = {
                    "signature_id": sid,
                    "signature_headers": list(sig),       # normalized canonical headers
                    "raw_headers_example": raw_headers,   # original style headers
                    "docs": set([doc_name]),
                    "pages": [(doc_name, page_num)],
                    "table_instances": 1,
                    "representative": {
                        "document_name": doc_name,
                        "page_num": page_num,
                        "table_index": t_idx,
                        "rows_sample": rows[:5],
                        "row_count": len(rows)
                    }
                }
            else:
                entry = unique_headers_catalog[sid]
                entry["docs"].add(doc_name)
                entry["pages"].append((doc_name, page_num))
                entry["table_instances"] += 1

                # keep the "best" representative by max row count
                if len(rows) > entry["representative"]["row_count"]:
                    entry["representative"] = {
                        "document_name": doc_name,
                        "page_num": page_num,
                        "table_index": t_idx,
                        "rows_sample": rows[:5],
                        "row_count": len(rows)
                    }

# Convert sets to lists for JSON friendliness
catalog_list = []
for sid, entry in unique_headers_catalog.items():
    catalog_list.append({
        "signature_id": entry["signature_id"],
        "signature_headers": entry["signature_headers"],
        "raw_headers_example": entry["raw_headers_example"],
        "docs_count": len(entry["docs"]),
        "docs": sorted(entry["docs"]),
        "table_instances": entry["table_instances"],
        "representative": entry["representative"]
    })

catalog_list.sort(key=lambda x: (-x["docs_count"], -x["table_instances"], x["signature_headers"]))

print(f"Unique header signatures: {len(catalog_list)}")
print("Top 10 unique headers:")
for i, c in enumerate(catalog_list[:10], start=1):
    print(f"\n[{i}] signature_id={c['signature_id'][:10]}... docs={c['docs_count']} instances={c['table_instances']}")
    print("Headers:", " | ".join(c["signature_headers"]))
    rep = c["representative"]
    print(f"Representative: {rep['document_name']} p{rep['page_num']} t{rep['table_index']} rows={rep['row_count']}")

# Optional save
out_catalog = os.path.join("data_processed", "unique_headers_catalog.json")
with open(out_catalog, "w", encoding="utf-8") as f:
    json.dump(catalog_list, f, ensure_ascii=False, indent=2)

print(f"\nSaved: {out_catalog}")

In [ ]:
for c in catalog_list:
    print(f"Raw Headers Example: {c['raw_headers_example']}")

In [ ]:
synonyms = {
    "eoc": ["EOC", "European Ordering Code", "ordering code", "order code", "commercial code", "sales code", "product order identifier"],
    "nc12": ["12NC", "12NC code", "12 digit numerical code", "12-digit product code", "Philips 12NC", "internal product code", "material number", "product id"],
    "box_quantity": ["box quantity", "pack size", "units per box", "packaging quantity", "carton quantity"],

    "operation_point": ["operation point", "operating point", "operating condition", "drive condition", "performance point", "current-temperature operating state"],

    "i_nom": ["I-nom", "nominal current", "rated current", "operating current"],
    "i_max": ["I-max", "maximum current", "peak current", "maximum drive current"],
    "i_life": ["I-life", "lifetime current", "recommended lifetime current"],
    "percent_i_nom": ["80% I-nom", "reduced current", "derated current"],

    "tc": ["Tc", "case temperature", "module temperature", "junction reference temperature"],
    "tc_nom": ["Tc-nom", "nominal case temperature", "rated temperature"],
    "tc_max": ["Tc-max", "maximum case temperature", "max operating temperature"],
    "tc_life": ["Tc life", "lifetime temperature", "temperature at lifetime condition"],

    "cct": ["CCT", "correlated color temperature", "color temperature", "kelvin rating"],
    "cri": ["CRI", "color rendering index", "Ra", "color fidelity"],
    "r9": ["R9", "deep red rendering index", "CRI R9"],
    "photometric_code": ["photometric code", "color code", "light color code", "930", "940", "color bin code"],
    "color_consistency": ["color consistency", "SDCM", "MacAdam ellipse", "color tolerance"],
    "cie": ["CIEx", "CIEy", "color coordinates", "chromaticity coordinates"],

    "luminous_flux": ["luminous flux", "flux", "light output", "lumens", "lm"],
    "efficacy": ["efficacy", "luminous efficacy", "efficiency", "lm/W", "lmw", "lm per watt"],

    "lm": ["lm", "lumens", "luminous flux"],
    "lm_per_w": ["lm/W", "lmw", "lm per watt", "luminous efficacy"],

    "flux_percent": ["flux %", "relative flux", "normalized flux"],
    "efficacy_percent": ["efficacy %", "relative efficacy"],

    "forward_voltage": ["forward voltage", "vf", "voltage drop", "led voltage"],
    "power": ["power consumption", "power", "wattage", "energy consumption"],
    "series_modules": ["modules in series", "series count", "modules per chain"],
    "parallel_modules": ["modules in parallel", "parallel count"],

    "l70": ["L70", "lumen maintenance 70%", "70% lifetime"],
    "l80": ["L80", "lumen maintenance 80%"],
    "l90": ["L90", "lumen maintenance 90%"],
    "b50": ["B50", "median failure rate"],
    "b10": ["B10", "10% failure"],
    "b20": ["B20", "20% failure"],
    "m70f50": ["M70F50", "median lifetime", "50% failure at 70% lumen"],

    "length": ["length", "module length"],
    "width": ["width"],
    "height_pcb": ["height pcb", "pcb thickness"],
    "height_connector": ["height incl connector", "total height"],
    "mass": ["mass", "weight", "product mass"],

    "esd": ["ESD", "electrostatic discharge"],
    "working_voltage": ["working voltage", "system voltage"],
    "storage_temp": ["storage temperature"],
    "certifications": ["CE", "ENEC", "ENEC+", "RoHS", "REACH"]
}

In [ ]:
import requests
import re
# These numeric codes are often used as column headers and can indicate photometric data by CCT, so we want to detect and normalize them specially
# They were obtained by looking at header values and brought here
CCT_CODES = {'827','830','835','840','842','850','865','927','930','935','940','950','965'}

OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL = "llama3.1:8b"

def classify_and_normalize_table(table):
    headers = table.get("headers", [])
    hset = set(headers)
    rows = table.get("rows", [])

    if "commercial_product_name" in hset:
        return "ordering_data", table

    cct_col = next((h for h in headers if h in CCT_CODES), None)
    if cct_col and "lm" in hset:
        new_headers = []
        for h in headers:
            if h in CCT_CODES:
                new_headers.append("tc_condition")
                new_headers.append("cct_code")
            elif h == "operation_window":
                new_headers.append("operation_point")
            else:
                new_headers.append(h)
        new_rows = []
        for row in rows:
            new_row = {}
            for h in headers:
                if h in CCT_CODES:
                    new_row["tc_condition"] = row.get(h, "")
                    new_row["cct_code"] = cct_col
                elif h == "operation_window":
                    new_row["operation_point"] = row.get(h, "")
                else:
                    new_row[h] = row.get(h, "")
            new_rows.append(new_row)
        return "photometric_by_cct", {"headers": new_headers, "rows": new_rows}

    if "flux_" in hset or "efficacy_" in hset:
        if any(h in hset for h in ("tc_c", "tcase_c")):
            new_headers = ["tc_c" if h in ("tc_c", "tcase_c") else h for h in headers]
            new_rows = [{("tc_c" if k in ("tc_c", "tcase_c") else k): v for k, v in row.items()} for row in rows]
            return "temperature_tuning", {"headers": new_headers, "rows": new_rows}
        if "i_ma" in hset:
            return "current_tuning", table

    if any(h in hset for h in ("l70", "l80", "l90", "lumen_maintenancebrx_1000_hours")):
        return "lumen_maintenance", table

    if "parameter" in hset and "unit" in hset:
        if "typ" in hset and "min" in hset:
            return "parameter_min_typ_max", table
        if "min" in hset:
            return "parameter_min_max", table
        if "nominal" in hset:
            return "parameter_nominal_life_max", table
        if "value" in hset:
            return "parameter_value", table
        if any(h.startswith("typical_") or h.startswith("average_") for h in headers):
            return "parameter_multi_temp", table
        return "parameter_other", table

    if "specification_item" in hset:
        return "wiring_spec", table

    return "unknown", table


# ── Classify all tables, collect sample rows per family ───────────────────────
family_counts = {}
family_samples = {}   # family -> {"headers": [...], "rows": [...up to 3...]}

for doc in documents:
    for page in doc.get("pages", []):
        for i, table in enumerate(page.get("tables", [])):
            family, normalized = classify_and_normalize_table(table)
            normalized["table_family"] = family
            page["tables"][i] = normalized
            family_counts[family] = family_counts.get(family, 0) + 1

            if family not in family_samples:
                sample_rows = [
                    {k: v for k, v in row.items() if k != "related_products"}
                    for row in normalized.get("rows", [])[:3]
                ]
                family_samples[family] = {
                    "headers": [h for h in normalized.get("headers", []) if h != "related_products"],
                    "rows": sample_rows
                }

print("Table family distribution:")
for family, count in sorted(family_counts.items(), key=lambda x: -x[1]):
    print(f"  {family:<35} {count:>5} tables")


# ── Ask Ollama for a semantic SQL table name per family ───────────────────────
def format_sample_rows(headers, rows):
    if not rows:
        return "  (no sample rows)"
    lines = ["  | " + " | ".join(headers) + " |"]
    lines.append("  |" + "|".join(["---"] * len(headers)) + "|")
    for row in rows:
        lines.append("  | " + " | ".join(str(row.get(h, "")) for h in headers) + " |")
    return "\n".join(lines)

def ask_semantic_table_name(family_key, headers, rows, used_names):
    forbidden = ", ".join(sorted(used_names)) or "none"
    sample_text = format_sample_rows(headers, rows)
    prompt = f"""You are naming SQL tables for an LED product datasheet catalog.

Table family key : {family_key}
Columns          : {", ".join(headers)}

Sample data:
{sample_text}

Already used names: {forbidden}

Rules:
- Return ONLY a single snake_case table name, nothing else — no explanation, no punctuation
- Must NOT be in the already used names list
- Must clearly describe what the data represents (e.g. optical_characteristics, ordering_data, lumen_maintenance)
- Maximum 5 words joined by underscores
- No prefix like t_ or tbl_

Table name:"""

    try:
        resp = requests.post(OLLAMA_URL, json={
            "model": MODEL,
            "prompt": prompt,
            "stream": False,
            "options": {"temperature": 0.0}
        }, timeout=30)
        resp.raise_for_status()
        raw = resp.json().get("response", "").strip().lower()
        name = re.sub(r"[^a-z0-9_]", "_", raw.split("\n")[0].strip())
        name = re.sub(r"_+", "_", name).strip("_")
        return name if name else None
    except Exception as e:
        print(f"  LLM error for '{family_key}': {e}")
        return None

def unique_name(candidate, used, fallback):
    if not candidate or candidate in used:
        candidate = fallback
    base, n = candidate, 2
    while candidate in used:
        candidate = f"{base}_{n}"
        n += 1
    return candidate


family_table_map = {}
used_table_names = set()

print("\nAsking Ollama for semantic table names...")
for family, sample in family_samples.items():
    suggested = ask_semantic_table_name(family, sample["headers"], sample["rows"], used_table_names)
    fallback  = re.sub(r"[^a-z0-9]", "_", family.lower())
    name      = unique_name(suggested, used_table_names, fallback)
    family_table_map[family] = name
    used_table_names.add(name)
    status = "(LLM)" if suggested == name else f"(suggested '{suggested}' → fallback)"
    print(f"  {family:<35} -> {name}  {status}")

# ── Stamp semantic_name onto every table in documents ─────────────────────────
for doc in documents:
    for page in doc.get("pages", []):
        for table in page.get("tables", []):
            family = table.get("table_family", "unknown")
            table["semantic_name"] = family_table_map.get(family, family)

print("\nsemantic_name stamped on all tables.")

In [ ]:
import sqlite3
import json
import os
import re

db_path = os.path.join("data_processed", "tables_catalog.db")
if os.path.exists(db_path):
    os.remove(db_path)

# ── Step 1: column union keyed by semantic_name (the actual SQL table name) ───
sql_table_columns = {}  # semantic_name -> ordered list of columns

for doc in documents:
    for page in doc.get("pages", []):
        for table in page.get("tables", []):
            sql_table = table.get("semantic_name")
            if not sql_table:
                raise ValueError(f"Table missing semantic_name — re-run classify cell. "
                                 f"Doc: {doc.get('document_name')}")
            if sql_table not in sql_table_columns:
                sql_table_columns[sql_table] = []
            for h in table.get("headers", []):
                if h not in sql_table_columns[sql_table]:
                    sql_table_columns[sql_table].append(h)

# ── Step 2: create schema ──────────────────────────────────────────────────────
conn = sqlite3.connect(db_path)
conn.execute("PRAGMA journal_mode=WAL")
conn.execute("PRAGMA foreign_keys=ON")
cur = conn.cursor()

cur.executescript("""
CREATE TABLE documents (
    document_name     TEXT PRIMARY KEY,
    targeted_products TEXT
);

CREATE TABLE table_instances (
    id                  INTEGER PRIMARY KEY AUTOINCREMENT,
    document_name       TEXT NOT NULL REFERENCES documents(document_name),
    page_num            INTEGER,
    table_index_on_page INTEGER,
    table_family        TEXT,
    table_name          TEXT
);

CREATE TABLE table_products (
    table_instance_id INTEGER NOT NULL REFERENCES table_instances(id),
    product_name      TEXT    NOT NULL,
    PRIMARY KEY (table_instance_id, product_name)
);

CREATE INDEX idx_ti_doc      ON table_instances(document_name);
CREATE INDEX idx_ti_family   ON table_instances(table_family);
CREATE INDEX idx_tp_product  ON table_products(product_name);
CREATE INDEX idx_tp_instance ON table_products(table_instance_id);
""")

print("Creating tables:")
for sql_table, cols in sql_table_columns.items():
    col_defs = ",\n    ".join(
        ["id INTEGER PRIMARY KEY AUTOINCREMENT",
         "table_instance_id INTEGER NOT NULL REFERENCES table_instances(id)"]
        + [f'"{c}" TEXT' for c in cols]
    )
    cur.execute(f'CREATE TABLE "{sql_table}" ({col_defs})')
    cur.execute(f'CREATE INDEX "idx_{sql_table}_ti" ON "{sql_table}"(table_instance_id)')
    print(f"  {sql_table:<45} ({len(cols)} cols)")

conn.commit()

# ── Step 3: insert ─────────────────────────────────────────────────────────────
for doc in documents:
    doc_name = doc["document_name"]
    cur.execute(
        "INSERT INTO documents(document_name, targeted_products) VALUES (?, ?)",
        (doc_name, doc.get("targeted_products", ""))
    )

    for page in doc.get("pages", []):
        page_num = page.get("page_num")
        for t_idx, table in enumerate(page.get("tables", []), start=1):
            family    = table.get("table_family", "unknown")
            sql_table = table.get("semantic_name")
            rows      = table.get("rows", [])
            all_cols  = sql_table_columns[sql_table]

            cur.execute(
                """INSERT INTO table_instances
                   (document_name, page_num, table_index_on_page, table_family, table_name)
                   VALUES (?, ?, ?, ?, ?)""",
                (doc_name, page_num, t_idx, family, sql_table)
            )
            ti_id = cur.lastrowid

            for row in rows:
                present      = {c: row[c] for c in all_cols if c in row}
                col_list     = ", ".join(f'"{c}"' for c in present)
                placeholders = ", ".join(["?"] * len(present))
                cur.execute(
                    f'INSERT INTO "{sql_table}" (table_instance_id, {col_list}) VALUES (?, {placeholders})',
                    [ti_id] + list(present.values())
                )

            related = rows[0].get("related_products", "") if rows else ""
            for product in [p.strip() for p in related.split(",") if p.strip()]:
                cur.execute(
                    "INSERT OR IGNORE INTO table_products(table_instance_id, product_name) VALUES (?, ?)",
                    (ti_id, product)
                )

conn.commit()
conn.close()
print(f"\nSaved: {db_path}")

# ── Step 4: verify ─────────────────────────────────────────────────────────────
conn = sqlite3.connect(db_path)
conn.row_factory = sqlite3.Row

print("\nAll tables:")
for row in conn.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"):
    count = conn.execute(f'SELECT COUNT(*) FROM "{row["name"]}"').fetchone()[0]
    print(f"  {row['name']:<45} {count:>6} rows")

print("\nFamily → semantic table name:")
for row in conn.execute("SELECT DISTINCT table_family, table_name FROM table_instances ORDER BY table_name"):
    print(f"  {row['table_family']:<35} -> {row['table_name']}")

conn.close()

In [ ]:
import sqlite3
import json
import os
import re

db_path = os.path.join("data_processed", "tables_catalog.db")
if os.path.exists(db_path):
    os.remove(db_path)

# ── Step 1: column union keyed by semantic_name ───────────────────────────────
sql_table_columns = {}

for doc in documents:
    for page in doc.get("pages", []):
        for table in page.get("tables", []):
            sql_table = table.get("semantic_name")
            if not sql_table:
                raise ValueError(f"Table missing semantic_name — re-run classify cell. "
                                 f"Doc: {doc.get('document_name')}")
            if sql_table not in sql_table_columns:
                sql_table_columns[sql_table] = []
            for h in table.get("headers", []):
                if h not in sql_table_columns[sql_table]:
                    sql_table_columns[sql_table].append(h)

# ── Step 2: create schema ──────────────────────────────────────────────────────
conn = sqlite3.connect(db_path)
conn.execute("PRAGMA journal_mode=WAL")
conn.execute("PRAGMA foreign_keys=ON")
cur = conn.cursor()

cur.executescript("""
CREATE TABLE documents (
    document_name     TEXT PRIMARY KEY,
    targeted_products TEXT
);

CREATE TABLE table_instances (
    id                  INTEGER PRIMARY KEY AUTOINCREMENT,
    document_name       TEXT NOT NULL REFERENCES documents(document_name),
    page_num            INTEGER,
    table_index_on_page INTEGER,
    table_family        TEXT,
    table_name          TEXT
);

CREATE TABLE table_products (
    table_instance_id INTEGER NOT NULL REFERENCES table_instances(id),
    product_name      TEXT    NOT NULL,
    PRIMARY KEY (table_instance_id, product_name)
);

CREATE INDEX idx_ti_doc      ON table_instances(document_name);
CREATE INDEX idx_ti_family   ON table_instances(table_family);
CREATE INDEX idx_tp_product  ON table_products(product_name);
CREATE INDEX idx_tp_instance ON table_products(table_instance_id);
""")

print("Creating tables:")
for sql_table, cols in sql_table_columns.items():
    col_defs = ",\n    ".join(
        ["id INTEGER PRIMARY KEY AUTOINCREMENT",
         "table_instance_id INTEGER NOT NULL REFERENCES table_instances(id)",
         "document_name TEXT",   # ← source document
         "page_num INTEGER"]     # ← source page
        + [f'"{c}" TEXT' for c in cols]
    )
    cur.execute(f'CREATE TABLE "{sql_table}" ({col_defs})')
    cur.execute(f'CREATE INDEX "idx_{sql_table}_ti" ON "{sql_table}"(table_instance_id)')
    cur.execute(f'CREATE INDEX "idx_{sql_table}_doc" ON "{sql_table}"(document_name)')
    print(f"  {sql_table:<45} ({len(cols)} data cols)")

conn.commit()

# ── Step 3: insert ─────────────────────────────────────────────────────────────
for doc in documents:
    doc_name = doc["document_name"]
    cur.execute(
        "INSERT INTO documents(document_name, targeted_products) VALUES (?, ?)",
        (doc_name, doc.get("targeted_products", ""))
    )

    for page in doc.get("pages", []):
        page_num = page.get("page_num")
        for t_idx, table in enumerate(page.get("tables", []), start=1):
            family    = table.get("table_family", "unknown")
            sql_table = table.get("semantic_name")
            rows      = table.get("rows", [])
            all_cols  = sql_table_columns[sql_table]

            cur.execute(
                """INSERT INTO table_instances
                   (document_name, page_num, table_index_on_page, table_family, table_name)
                   VALUES (?, ?, ?, ?, ?)""",
                (doc_name, page_num, t_idx, family, sql_table)
            )
            ti_id = cur.lastrowid

            for row in rows:
                present      = {c: row[c] for c in all_cols if c in row}
                col_list     = ", ".join(f'"{c}"' for c in present)
                placeholders = ", ".join(["?"] * len(present))
                cur.execute(
                    f'INSERT INTO "{sql_table}" '
                    f'(table_instance_id, document_name, page_num, {col_list}) '
                    f'VALUES (?, ?, ?, {placeholders})',
                    [ti_id, doc_name, page_num] + list(present.values())
                )

            related = rows[0].get("related_products", "") if rows else ""
            for product in [p.strip() for p in related.split(",") if p.strip()]:
                cur.execute(
                    "INSERT OR IGNORE INTO table_products(table_instance_id, product_name) VALUES (?, ?)",
                    (ti_id, product)
                )

conn.commit()
conn.close()
print(f"\nSaved: {db_path}")

# ── Step 4: verify ─────────────────────────────────────────────────────────────
conn = sqlite3.connect(db_path)
conn.row_factory = sqlite3.Row

print("\nAll tables:")
for row in conn.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"):
    count = conn.execute(f'SELECT COUNT(*) FROM "{row["name"]}"').fetchone()[0]
    print(f"  {row['name']:<45} {count:>6} rows")

print("\nFamily → semantic table name:")
for row in conn.execute("SELECT DISTINCT table_family, table_name FROM table_instances ORDER BY table_name"):
    print(f"  {row['table_family']:<35} -> {row['table_name']}")

conn.close()